# EvaLoop: standard vs. multilingual summary loop (OpenRouter on Colab)

Runs every (model, loop) configuration through OpenRouter and tests the generated code here, on Colab's
temporary machine, never on your own computer.

**Before you start**

1. *Runtime → Change runtime type → CPU* (no GPU needed; the models run at OpenRouter).
2. Add your OpenRouter key: click the **key icon (Secrets)** in the left sidebar, add a secret named
   `OPENROUTER_API_KEY`, paste the key as its value, and switch on *Notebook access*. Never paste the key
   into a cell.
3. **Don't mount Google Drive** in this notebook: the generated code is untrusted. Each finished run is
   downloaded to your computer automatically instead.
4. Keep this tab open while it runs. If anything stops it (network, credits, disconnect), just run the
   *Run the experiments* cell again: finished runs are skipped and an interrupted run continues from its
   last completed cycle (as long as the Colab machine wasn't reset).

## 1. Settings

In [ ]:
FORK_URL = "https://github.com/ankit-bhade/EvaLooop.git"  # your public fork
BRANCH = "multilingual-summary"

# OpenRouter model id -> provider to pin, so every run of a model uses the same backend
MODELS = {
    "qwen/qwen3-coder-30b-a3b-instruct": "amazon-bedrock",
    "google/gemma-3-12b-it": "deepinfra/bf16",
}
LANGUAGES = ["english", "spanish"]  # "english" = the standard loop (the baseline)

MAX_TASKS = 10      # quick test run; set to None for the real runs (all 378 tasks)
SUBSET_SEED = 0     # same seed -> same tasks for every configuration
MAX_CYCLES = 10
CONCURRENCY = 16    # parallel requests to OpenRouter

## 2. Install

In [ ]:
import os, subprocess

os.chdir("/content")
if not os.path.exists("EvaLooop"):
    subprocess.run(["git", "clone", "-b", BRANCH, FORK_URL, "EvaLooop"], check=True)
os.chdir("/content/EvaLooop")
subprocess.run(["git", "pull", "--ff-only"], check=True)
subprocess.run(["pip", "-q", "install", "-e", "."], check=True)
print("Installed at commit:", subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                                             capture_output=True, text=True).stdout.strip())

## 3. Load the API key and check both models answer

Costs a fraction of a cent.

In [ ]:
from google.colab import userdata
from openai import OpenAI

os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])

for model_id, provider in MODELS.items():
    reply = client.chat.completions.create(
        model=model_id,
        messages=[{"role": "user", "content": "Reply with the single word OK."}],
        max_tokens=5,
        temperature=0,
        extra_body={"provider": {"order": [provider], "allow_fallbacks": False}},
    )
    served_by = getattr(reply, "provider", None) or (reply.model_extra or {}).get("provider")
    print(f"{model_id}: {reply.choices[0].message.content!r} (served by {served_by})")

## 4. Run the experiments

Runs one configuration at a time (running several at once can overload Colab's 2 CPUs and make correct code time out in the tests).

In [ ]:
from google.colab import files

def run_name(model_id, language):
    subset = f"_n{MAX_TASKS}_seed{SUBSET_SEED}" if MAX_TASKS else ""
    return f"{model_id.split('/')[-1]}_{language}{subset}"

for model_id, provider in MODELS.items():
    for language in LANGUAGES:
        name = run_name(model_id, language)
        results_file = f"results/{name}_code_generation_summarization_results.json"
        if os.path.exists(results_file):
            print("already done:", name)
            continue
        cmd = ["evaloop", "evaluate",
               "--model", model_id,
               "--api_base_url", "https://openrouter.ai/api/v1",
               "--api_key_env", "OPENROUTER_API_KEY",
               "--api_provider", provider,
               "--api_concurrency", str(CONCURRENCY),
               "--summary_language", language,
               "--max_cycles", str(MAX_CYCLES),
               "--temperature", "0",
               "--experiment_name", name,
               "--output_dir", "results/",
               "--log_level", "WARNING",
               "--resume"]
        if MAX_TASKS:
            cmd += ["--max_tasks", str(MAX_TASKS), "--subset_seed", str(SUBSET_SEED)]
        print("running:", name)
        subprocess.run(cmd, check=True)
        files.download(results_file)

## 5. Summary table

`asl_score` is ASL_pow (the judge-free ASL, the leaderboard's `aslScore`); `success_rate` is pass@1.

In [ ]:
import glob

results = sorted(glob.glob("results/*_results.json"))
subprocess.run(["python", "scripts/summarize_runs.py", *results, "--output", "summary.csv"], check=True)
subprocess.run(["zip", "-j", "-q", "results.zip", *results], check=True)
files.download("summary.csv")
files.download("results.zip")